# Evaluation Consistency & Reliability Demo

**Question this notebook answers:** *If I run the exact same evaluation, on the exact same agent session, several times — do I get the same score?*

This is the **consistency** (a.k.a. **reliability** or **stability**) of an evaluator. It matters because most
AgentCore evaluators are **LLM-as-a-Judge**: a language model reads the agent's output and scores it. Like any
LLM call, that judgement is *non-deterministic* — the same input can yield different scores on different runs.

> **Self-contained:** this notebook **deploys its own** City Search Agent to AgentCore Runtime from the
> `citysearch.py` file in this directory. It does not depend on variables `%store`d by any other notebook, so
> you can run it on its own.

### The experiment

1. Deploy the City Search Agent and create **one** fixed session (a single New York query).
2. Run the **identical** set of evaluators against that **same session** `N_RUNS` times.
3. Collect every score and measure the spread per evaluator.
4. Visualize how each evaluator behaves across runs.

### What you'll see

| Evaluator kind | Example | Expected behaviour across runs |
|---|---|---|
| **LLM-as-a-Judge** (stochastic) | `Correctness`, `Helpfulness`, `GoalSuccessRate` | Scores vary — a *distribution*, not a point |
| **Programmatic** (deterministic) | `TrajectoryExactOrderMatch`, `TrajectoryInOrderMatch` | Identical value every run — a flat line |

That contrast is the whole point: it tells you **how much to trust a single evaluation score**, and which
metrics need averaging over multiple runs before you draw conclusions.

### Prerequisites
- Python 3.10+
- AWS credentials with permissions for AgentCore, CloudWatch, ECR, IAM, CodeBuild
- `citysearch.py` present in this directory (it is — created by notebook `01`, and checked in here)

## Step 1: Install Dependencies

In [ ]:
!pip install -r requirements.txt -q

## Step 2: Configuration

Import libraries and pick up the region from your AWS session.

In [ ]:
import json
import time
import uuid
from datetime import timedelta

import boto3
import numpy as np
import pandas as pd
from boto3.session import Session

boto_session = Session()
REGION = boto_session.region_name

print(f"Region : {REGION}")

## Step 3: Deploy the City Search Agent

We deploy the City Search Agent to **AgentCore Runtime** from the `citysearch.py` file that already lives in
this directory. Creating the agent here — rather than restoring `%store`d identifiers from notebook `01` —
keeps this notebook runnable on its own.

The agent uses:
- **web_search tool**: Queries DuckDuckGo for current city population and area data
- **XML output format**: `<pop>` and `<area>` tags for programmatic parsing

The deployment steps are:
1. **Verify** — confirm `citysearch.py` is present
2. **Configure** — set up ECR, IAM roles, and agent configuration
3. **Launch** — build the container via CodeBuild and deploy to AgentCore Runtime

> Re-running this notebook updates the same runtime in place (`auto_update_on_conflict=True`) instead of
> creating a duplicate.

In [ ]:
import os

# The agent entrypoint ships with this module — fail fast if it is missing.
AGENT_FILE = "citysearch.py"
if not os.path.exists(AGENT_FILE):
    raise FileNotFoundError(
        f"'{AGENT_FILE}' not found in {os.getcwd()}. "
        "Run '01 AgentCore Evals Ground Truth.ipynb' (Step 3) to write it, or restore it from the repo."
    )

print(f"Found {AGENT_FILE} ({os.path.getsize(AGENT_FILE)} bytes).")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

AGENT_NAME = "citysearch_groundtruth_eval"

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint=AGENT_FILE,
    agent_name=AGENT_NAME,
    region=REGION,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    non_interactive=True,
)
print("Configuration complete.")

print("\nDeploying City Search Agent ...")
print("  This takes ~5 minutes on first run (image build + push + runtime creation).")
print()

_launch = agentcore_runtime.launch(auto_update_on_conflict=True)

print(f"\nLaunch complete.")
print(f"  agent_id  : {_launch.agent_id}")
print(f"  agent_arn : {_launch.agent_arn}")

In [ ]:
print("Waiting for agent to reach READY status ...")

_POLL_INTERVAL = 15   # seconds between status checks
_MAX_WAIT      = 600  # 10-minute timeout

_elapsed = 0
while _elapsed < _MAX_WAIT:
    _status_result = agentcore_runtime.status()
    _agent_info    = _status_result.agent or {}
    _agent_status  = _agent_info.get("status", "UNKNOWN")
    print(f"  [{_elapsed:>3}s] status = {_agent_status}")

    if _agent_status in ("READY", "ACTIVE"):
        print(f"\nAgent is {_agent_status}. Proceeding.")
        break
    if _agent_status in ("FAILED", "CREATE_FAILED", "UPDATE_FAILED"):
        raise RuntimeError(
            f"Agent deployment failed with status '{_agent_status}'.\n"
            f"Details: {_agent_info}"
        )

    time.sleep(_POLL_INTERVAL)
    _elapsed += _POLL_INTERVAL
else:
    raise TimeoutError(
        f"Agent did not reach READY status within {_MAX_WAIT}s. "
        "Check the AgentCore console for details."
    )

In [ ]:
from botocore.config import Config

AGENT_ID     = _launch.agent_id
AGENT_ARN    = _launch.agent_arn
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")
print(f"REGION       : {REGION}")

agentcore_config = Config(read_timeout=180, retries={"max_attempts": 2})
agentcore_client = boto3.client(
    "bedrock-agentcore", region_name=REGION, config=agentcore_config
)

## Step 4: Create One Fixed Session

We invoke the agent **once** with a single query and keep its `session_id`. Every evaluation run in this
notebook targets *this one session* — so any variation we observe comes purely from the evaluators, never
from the agent producing a different answer.

We also supply ground truth (`expected_response`, `expected_trajectory`, `assertions`) so the ground-truth
evaluators have something to compare against.

In [ ]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """Send one prompt to the city search agent and return its text response."""
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}),
    )
    response_data = json.loads(resp["response"].read())
    return response_data if isinstance(response_data, str) else str(response_data)


# --- The single, fixed input session everything is evaluated against ---
PROMPT = "How many people live in New York, and what's the area of the city in square miles?"
CONSISTENCY_SESSION_ID = f"consistency-demo-{uuid.uuid4()}"

print(f"Session : {CONSISTENCY_SESSION_ID}")
print(f"  > {PROMPT}")
_response = invoke_agent(PROMPT, CONSISTENCY_SESSION_ID)
print(f"  < {_response[:200]}")

print("\nWaiting 90s for CloudWatch span ingestion before evaluating ...")
time.sleep(90)
print("Ready to evaluate.")

## Step 5: Configure the Repeated Evaluation

We deliberately mix **stochastic** LLM-judge evaluators with **deterministic** trajectory matchers so the
difference in consistency is visible in the same chart.

`N_RUNS` controls how many times we repeat the *identical* evaluation. More runs → a tighter estimate of each
evaluator's true variance, but more LLM-judge calls (cost + time). 8–12 is plenty to see the pattern.

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

eval_client = EvaluationClient(region_name=REGION)

# How many times to repeat the identical evaluation against the same session.
N_RUNS = 5

# Mix of stochastic (LLM-judge) and deterministic (programmatic) evaluators.
EVALUATOR_IDS = [
    "Builtin.Correctness",                # TRACE   — LLM-judge (stochastic)
    "Builtin.Helpfulness",                # TRACE   — LLM-judge (stochastic)
    "Builtin.GoalSuccessRate",            # SESSION — LLM-judge (stochastic)
    "Builtin.TrajectoryExactOrderMatch",  # SESSION — programmatic (deterministic)
    "Builtin.TrajectoryInOrderMatch",     # SESSION — programmatic (deterministic)
]

# Ground truth for the New York session.
REFERENCE_INPUTS = ReferenceInputs(
    expected_response=(
        "New York has a population of approximately 8,478,072 people "
        "and a land area of about 300.5 square miles."
    ),
    expected_trajectory=["web_search"],
    assertions=[
        "Agent called web_search to look up New York population and area",
        "Agent reported a population close to 8,478,072",
        "Agent reported a land area close to 300.5 square miles",
    ],
)

print(f"Will run {len(EVALUATOR_IDS)} evaluators x {N_RUNS} runs "
      f"= {len(EVALUATOR_IDS) * N_RUNS} evaluations against 1 fixed session.")

## Step 6: Run the Same Evaluation `N_RUNS` Times

Each iteration calls `eval_client.run(...)` with **identical** arguments against the **same** `session_id`.
We tag every returned score with its run number and collect them into a tidy list.

In [ ]:
def short_name(evaluator_id: str) -> str:
    """Strip the 'Builtin.' prefix for compact display."""
    return evaluator_id.split(".")[-1]


def extract_value(result: dict):
    """Pull the numeric score from an evaluator result dict (None on error/missing)."""
    if result.get("errorCode"):
        return None
    v = result.get("value", result.get("score"))
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None


records = []
for run_idx in range(1, N_RUNS + 1):
    print(f"Run {run_idx}/{N_RUNS} ...", end=" ", flush=True)
    try:
        results = eval_client.run(
            evaluator_ids=EVALUATOR_IDS,
            session_id=CONSISTENCY_SESSION_ID,
            agent_id=AGENT_ID,
            look_back_time=timedelta(hours=2),
            reference_inputs=REFERENCE_INPUTS,
        )
    except Exception as e:  # keep the loop alive if one run fails
        print(f"FAILED: {e}")
        continue

    scored = []
    for r in results:
        value = extract_value(r)
        records.append({
            "run": run_idx,
            "evaluator": short_name(r.get("evaluatorId", "")),
            "value": value,
            "label": r.get("label", r.get("rating", "")),
        })
        if value is not None:
            scored.append(f"{short_name(r.get('evaluatorId',''))}={value:.2f}")
    print(", ".join(scored) if scored else "no scores")

df = pd.DataFrame(records)
print(f"\nCollected {len(df)} score records across {df['run'].nunique()} runs.")
df.head(len(EVALUATOR_IDS))

## Step 7: Consistency Metrics

For each evaluator we summarize the distribution of scores across runs:

| Metric | Meaning |
|---|---|
| **mean** | Average score — the number you'd report from a single run *in expectation* |
| **std** | Standard deviation — the raw spread. `0.0` = perfectly consistent |
| **range** | max − min — worst-case swing between two runs |
| **CV %** | Coefficient of variation (`std / mean`) — spread relative to the score, comparable across evaluators |
| **n_unique** | Number of distinct values seen. `1` = deterministic |
| **deterministic** | `True` when every run returned the same value |

A low std / CV means you can trust a single run. A high std means you should **average multiple runs** before
acting on the score.

In [ ]:
valid = df.dropna(subset=["value"])

summary = (
    valid.groupby("evaluator")["value"]
    .agg(runs="count", mean="mean", std="std", min="min", max="max", n_unique="nunique")
)
summary["std"] = summary["std"].fillna(0.0)
summary["range"] = summary["max"] - summary["min"]
summary["cv_%"] = np.where(
    summary["mean"] != 0, summary["std"] / summary["mean"] * 100, 0.0
)
summary["deterministic"] = summary["n_unique"] <= 1
summary = summary.sort_values("std", ascending=False)

# Order columns for readability
summary = summary[["runs", "mean", "std", "range", "cv_%", "min", "max", "n_unique", "deterministic"]]
summary.round(3)

In [ ]:
# A one-line verdict per evaluator
print("Consistency verdict (same session, repeated evaluation)")
print("=" * 62)
for name, row in summary.iterrows():
    if row["deterministic"]:
        verdict = "DETERMINISTIC  — identical every run"
    elif row["std"] < 0.05:
        verdict = "very consistent"
    elif row["std"] < 0.15:
        verdict = "moderately variable — consider averaging"
    else:
        verdict = "HIGH variance — average several runs before trusting"
    print(f"  {name:<28} mean={row['mean']:.2f}  std={row['std']:.3f}  →  {verdict}")

## Step 8: Visualize — Scores Across Runs

One line per evaluator, score on the y-axis, run number on the x-axis. Deterministic evaluators appear as
**flat lines**; stochastic LLM-judges **wobble**. This is the picture to show in the demo.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# --- Validated categorical palette (light mode) + chart chrome ---
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "font.family": "sans-serif", "font.size": 11,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
})

pivot = valid.pivot_table(index="run", columns="evaluator", values="value")
# Stable order: most-variable evaluators (from summary) first
pivot = pivot[[c for c in summary.index if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(9, 5.2))
for i, col in enumerate(pivot.columns):
    is_det = bool(summary.loc[col, "deterministic"])
    ax.plot(
        pivot.index, pivot[col],
        color=SERIES[i % len(SERIES)],
        linewidth=2, marker="o", markersize=6,
        linestyle="--" if is_det else "-",
        label=f"{col}{'  (deterministic)' if is_det else ''}",
    )

ax.set_ylim(-0.05, 1.08)
ax.set_xlabel("Evaluation run (same session, same inputs)")
ax.set_ylabel("Score")
ax.set_title("Evaluator scores across repeated runs of one fixed session", color=INK, fontsize=13, pad=12)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="y", color=GRID, linewidth=1)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=2)
fig.tight_layout()
plt.show()

## Step 9: Visualize — How Variable Is Each Evaluator?

A single bar per evaluator showing its standard deviation across runs. Taller bar = less consistent =
needs more runs to trust. Zero-height bars are the deterministic evaluators.

In [ ]:
order = summary.sort_values("std", ascending=True)
fig, ax = plt.subplots(figsize=(9, 0.6 * len(order) + 1.5))

bars = ax.barh(order.index, order["std"], color="#2a78d6", height=0.6, zorder=3)
# Round the data-end of each bar's baseline (thin marks, direct labels)
for name, val in zip(order.index, order["std"]):
    ax.text(val + 0.004, name, f"{val:.3f}", va="center", ha="left", color=INK, fontsize=10)

ax.set_xlim(0, max(0.12, order["std"].max() * 1.25))
ax.set_xlabel("Standard deviation of score across runs  (0 = perfectly consistent)")
ax.set_title("Evaluator consistency — lower is better", color=INK, fontsize=13, pad=12)
ax.grid(axis="x", color=GRID, linewidth=1)
ax.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="y", length=0, labelcolor=INK)
fig.tight_layout()
plt.show()

## Step 10: Save the Consistency Report

In [ ]:
import os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
report_path = f"results/consistency_report_{timestamp}.json"

report = {
    "session_id": CONSISTENCY_SESSION_ID,
    "agent_id": AGENT_ID,
    "prompt": PROMPT,
    "n_runs": int(df["run"].nunique()),
    "evaluator_ids": EVALUATOR_IDS,
    "per_run_scores": records,
    "summary": summary.round(4).reset_index().to_dict(orient="records"),
}
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Consistency report saved to: {report_path}")

## Key Takeaways

- **LLM-as-a-Judge evaluators are non-deterministic.** The same session, scored repeatedly, yields a
  *distribution* of scores — not a single fixed number. `Correctness`, `Helpfulness`, and `GoalSuccessRate`
  are the ones to watch.
- **Programmatic evaluators are deterministic.** `TrajectoryExactOrderMatch` / `TrajectoryInOrderMatch`
  return the identical value on every run (flat line, std = 0).
- **std and CV quantify trust.** A near-zero std means a single run is reliable; a high std means you should
  **average several runs** (or raise the judge model / tighten the rubric) before acting on the score.
- **This is why regression gates should use margins, not exact equality.** If `Correctness` has std ≈ 0.1,
  a drop from 0.80 to 0.75 between two CI runs may be noise — not a real regression.

### Knobs to explore

| Change | Effect |
|---|---|
| Increase `N_RUNS` | Tighter estimate of true variance (more cost) |
| Swap the judge `modelId` for a larger model | Often reduces variance |
| Use a binary pass/fail rating scale instead of 0/0.5/1.0 | Fewer possible values → usually more consistent |
| Add your own custom evaluators from notebook `01` | Compare their stability against the built-ins |

## Cleanup

This notebook deployed its own agent, so remove it when you're done to avoid ongoing charges for AgentCore
Runtime compute, ECR container storage, and CloudWatch log storage.

In [ ]:
# Destroy the agent deployed by this notebook.
#
# WARNING: This action is irreversible — re-run Step 3 to deploy again.

print("Agent deployed by this notebook")
print("=" * 60)
print(f"  agent_name : {AGENT_NAME}")
print(f"  agent_arn  : {AGENT_ARN}")
print("=" * 60)

# Uncomment to actually destroy the agent:
# destroy_result = agentcore_runtime.destroy()
# print(f"\nAgent destroyed: {destroy_result}")

print("\nAgent destruction is commented out for safety.")
print("To destroy, uncomment the lines above and re-run this cell, or from the command line:")
print(f"  agentcore destroy --agent-name {AGENT_NAME}")